# Stemming, Lemmatisation, and Why Preprocessing Is Not Neutral

**Computational Text Analysis Pack · Module 14 — Preparing Text, Properly**

*Technique:* Suffix stripping vs dictionary lookup; Schofield & Mimno's warning

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sagaustus/adh-dh-notebooks/blob/main/01-preparing-text/14a_stemming_and_lemmatisation.ipynb)

Data: `masakhanews.csv`, `colonial_texts.csv` — from the `data/` folder of this pack.

---

In [ ]:
# Setup — run this first.
import os, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
from collections import Counter

# The data this notebook needs. It is downloaded from CourseHub automatically;
# uploading by hand is only a fallback if that server is unreachable.
NEEDED = ['masakhanews.csv', 'colonial_texts.csv']
DATA_URL = 'https://coursehub.africandigitalhumanities.ca/ide/pack-data/text/'

def _missing():
    return [f for f in NEEDED if not os.path.exists(f)]

missing = _missing()
if missing:
    # Fetch from CourseHub first, so nobody has to upload anything.
    import urllib.request, urllib.parse
    for _f in list(missing):
        try:
            urllib.request.urlretrieve(DATA_URL + urllib.parse.quote(_f), _f)
            print(f'downloaded {_f}')
        except Exception as _e:
            print(f'could not fetch {_f}: {_e}')
    missing = _missing()

if missing:
    try:
        from google.colab import files
    except ImportError:
        raise SystemExit('Place these next to the notebook: ' + ', '.join(missing))
    # Ask again until everything has arrived. The upload widget returns as soon
    # as you close it, so picking only some of the files would otherwise fail a
    # few lines below with a confusing FileNotFoundError.
    for _ in range(4):
        print('Select ALL of these at once (ctrl-click / cmd-click to multi-select):')
        print('   ' + ', '.join(missing))
        files.upload()
        missing = _missing()
        if not missing:
            break
        print('Still needed: ' + ', '.join(missing))
    if missing:
        raise SystemExit(
            'Missing: ' + ', '.join(missing) + '. Re-run this cell and select '
            'every file listed, or upload them with the folder icon on the left.')

df       = pd.read_csv('masakhanews.csv')
df_colonial = pd.read_csv('colonial_texts.csv')
NIGERIAN = ['Hausa', 'Igbo', 'Yoruba', 'Nigerian Pidgin']
# Most of this pack runs on scikit-learn and the standard library. Also
# available: nltk, spacy, gensim and networkx (modules 14-19), Keras
# (case studies), and a GPU pod reached through /ide/api/neural/ for the
# neural modules — never called directly, so no token lives in a notebook.

print(f'Loaded. df has {len(df):,} rows and {df.shape[1]} columns.')

## What this notebook does

Two operations reduce a word to a root form, and they are not the same thing.

- **Stemming** chops suffixes off by rule. *Studies* becomes *studi*. The output
  is often not a word, and the stemmer neither knows nor cares.
- **Lemmatisation** maps a word to its dictionary head-form. *Studies* becomes
  *study*, *better* becomes *good*. It needs a lexicon and morphological
  knowledge, which is why it is slower and language-specific.

Then the finding that should change how you use either: Schofield and Mimno
(2016) showed stemming can make topic models **worse**, not better.

## The tooling

**NLTK (Natural Language Toolkit)** is the academic textbook of Python NLP:
sentence splitting, punctuation stripping, stemming, and a large collection of
classic algorithms implemented for teaching rather than speed. It is the right
tool for traditional text preparation and slow on very large corpora.

**spaCy** is the industrial alternative — built for throughput, and grammatical
rather than lexical: it parses sentence structure, so it can tell the *bank* of
a river from the *bank* that holds money. It ships English-first models, which
matters here more than it does in most courses.

We write the stemmer by hand first because the survey's point about stemming is
that it is a crude rule-based truncation, and you only feel that by writing the
rules.

In [ ]:
import re

def tokenise(text):
    """Words, lowercased, Unicode-aware.

    `\\w+` matches Unicode letters, so ọ, à, ƙ, ɗ and ɓ survive. The common
    shortcut `[a-z]+` would throw them away and quietly change every count in
    this notebook — see notebook 03b.
    """
    return re.findall(r"\w+", str(text).lower())

sample = "Àwọn ọmọ ìlú ti ṣe ìpàdé — ƙungiyar ɗalibai ta yi taro"
print("sample     :", sample)
print("tokens     :", tokenise(sample))
print("naive [a-z]:", re.findall(r"[a-z]+", sample.lower()))
print()
print("The naive pattern splits words at every accented letter.")

In [ ]:
# A minimal Porter-style stemmer, written out so the rules are visible.
def measure(stem):
    """Porter's m: the number of vowel-consonant sequences."""
    vowels = "aeiou"
    form = "".join("V" if c in vowels else "C" for c in stem)
    return form.count("VC")

def simple_stem(word):
    for suffix, replacement in [
        ("ational", "ate"), ("tional", "tion"), ("ization", "ize"),
        ("fulness", "ful"), ("ousness", "ous"), ("iveness", "ive"),
        ("ies", "i"), ("sses", "ss"), ("ing", ""), ("edly", ""),
        ("ed", ""), ("ly", ""), ("s", ""),
    ]:
        if word.endswith(suffix):
            stem = word[: -len(suffix)] + replacement
            if measure(stem) > 0:
                return stem
    return word

for w in ["studies", "studying", "national", "organization", "happiness",
          "flies", "running", "quickly", "was"]:
    print(f"  {w:<14} -> {simple_stem(w)}")
print()
print("Note 'studi', 'organize', 'happi'. Several are not words at all — that is")
print("expected. A stemmer's job is to collapse variants, not to produce English.")

## The same words through NLTK and spaCy

Now the libraries. NLTK's Porter stemmer is the reference implementation of the
rules above; its WordNet lemmatiser does the dictionary-lookup job instead.

In [ ]:
import nltk
from nltk.stem import PorterStemmer, WordNetLemmatizer

porter = PorterStemmer()
lemmatiser = WordNetLemmatizer()

print(f"{'word':<14}{'ours':<12}{'NLTK Porter':<14}{'NLTK lemma':<12}")
print("-" * 52)
for w in ["studies", "studying", "national", "organization", "happiness",
          "flies", "running", "better", "was", "children"]:
    print(f"{w:<14}{simple_stem(w):<12}{porter.stem(w):<14}"
          f"{lemmatiser.lemmatize(w, pos='v'):<12}")
print()
print("Lemmatisation knows 'was' -> 'be' and 'children' -> 'child'. No suffix")
print("rule can reach those, because the relationship is stored, not derived.")

## What spaCy adds: grammar

A lemmatiser needs to know a word's part of speech — *saw* is the past of *see*
or a tool, depending on the sentence. spaCy parses the sentence first, which is
the difference between a lexical tool and a grammatical one.

In [ ]:
import spacy

nlp = spacy.load("en_core_web_sm")
for sentence in ["I saw the carpenter yesterday.",
                 "He reached for the saw on the bench."]:
    doc = nlp(sentence)
    print(sentence)
    print("   " + "  ".join(f"{t.text}/{t.pos_}->{t.lemma_}" for t in doc
                            if not t.is_punct))
print()
print("Same surface form, different lemma, because the parse differs.")
print("A stemmer cannot make this distinction; it never sees the sentence.")

## Schofield and Mimno: stemming can make things worse

The intuition is that collapsing *study/studies/studying* gives a topic model
cleaner evidence. Schofield and Mimno tested it and found stemmers often
**degrade** model quality, because they conflate words that mean different
things and do so inconsistently.

We can reproduce the shape of that result: fit the same topic model twice, once
on raw tokens and once on stemmed ones, and compare coherence.

In [ ]:
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.feature_extraction.text import CountVectorizer

sub = df[df["language"] == "English"].reset_index(drop=True)
raw_docs = [" ".join(tokenise(t)) for t in sub["text"]]
stemmed_docs = [" ".join(porter.stem(w) for w in tokenise(t)) for t in sub["text"]]

def coherence_of(docs, K=8):
    vec = CountVectorizer(min_df=5, max_df=0.4, max_features=2000)
    X = vec.fit_transform(docs)
    lda = LatentDirichletAllocation(n_components=K, random_state=0, max_iter=20)
    lda.fit(X)
    binary = (X > 0).astype(int)
    n_docs, scores = X.shape[0], []
    df_counts = np.asarray(binary.sum(axis=0)).ravel()
    for k in range(K):
        top = np.argsort(lda.components_[k])[::-1][:10]
        for a in range(len(top)):
            for b in range(a + 1, len(top)):
                i, j = top[a], top[b]
                co = int(binary[:, i].multiply(binary[:, j]).sum())
                if co == 0:
                    scores.append(-1.0)
                    continue
                p_ij = co / n_docs
                scores.append(np.log(p_ij / ((df_counts[i]/n_docs) * (df_counts[j]/n_docs)))
                              / -np.log(p_ij))
    return float(np.mean(scores)), vec

raw_coh, raw_vec = coherence_of(raw_docs)
stem_coh, stem_vec = coherence_of(stemmed_docs)
print(f"vocabulary, raw     : {len(raw_vec.get_feature_names_out()):,}")
print(f"vocabulary, stemmed : {len(stem_vec.get_feature_names_out()):,}")
print()
print(f"mean NPMI coherence, raw     : {raw_coh:.4f}")
print(f"mean NPMI coherence, stemmed : {stem_coh:.4f}")
print()
print("Whichever way this lands on your run, the lesson is that it is not")
print("automatic. Stemming is a modelling decision with a measurable cost, and")
print("'we stemmed the corpus' is not a neutral sentence in a methods section.")

In [ ]:
# Which distinctions did the stemmer destroy?
groups = {}
for w in raw_vec.get_feature_names_out():
    groups.setdefault(porter.stem(w), []).append(w)
collapsed = {s: ws for s, ws in groups.items() if len(ws) > 2}
print(f"{len(collapsed)} stems absorb three or more distinct words. A sample:")
for stem, words in list(sorted(collapsed.items(), key=lambda kv: -len(kv[1])))[:10]:
    print(f"  {stem:<14} <- {', '.join(words[:7])}")
print()
print("Scan that list for pairs that do NOT mean the same thing. Those are the")
print("conflations that cost you, and they are invisible after the fact.")

### 🔧 Your turn 1

Run the coherence comparison on the colonial corpus instead (`df_colonial`). Does stemming help or hurt there?

<details markdown="1">
<summary><b>What you should have seen</b> — click to expand</summary>

Results differ by corpus, which is the actual finding. The colonial corpus is
longer-form prose with a wider vocabulary, so stemming collapses more and the
effect is usually larger in whichever direction it runs. Schofield and Mimno's
conclusion was not "never stem" but "do not assume"; the defensible practice is
to run it both ways on *your* corpus and report which you chose and why. Note
also that none of this transfers to the Nigerian-language documents at all —
Porter is a set of English suffix rules, and applying it to Hausa or Yorùbá is
not a weaker version of the same operation, it is a different and wrong one.

</details>